# Static alphabet classifier — independent test evaluation
Train/validation/test are separate. Test data never controls fitting or early stopping. Legacy signer/session IDs are unavailable; see training/PROVENANCE.md.

In [1]:
import os
from pathlib import Path
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from training.data import load_dataset, load_groups, split_indices, sample_hash
from sklearn.utils.class_weight import compute_class_weight
import json
import hashlib

ROOT = Path.cwd()
assert (ROOT / "model/keypoint_classifier/keypoint.csv").exists(), "Run from the project root"
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(2)
tf.keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()
MODEL_DIR = ROOT / "model/keypoint_classifier"
model_save_path = MODEL_DIR / "keypoint_classifier.keras"
tflite_save_path = MODEL_DIR / "keypoint_classifier.tflite"


/Users/a/Speech-Detection/sign_language/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
X, y = load_dataset(MODEL_DIR / "keypoint.csv")
groups = load_groups(MODEL_DIR / "sample_metadata.csv", X, y)
train_idx, val_idx, test_idx = split_indices(y, groups)
X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]
method = "session-grouped" if groups is not None else "stratified-sample"
limitation = None if groups is not None else "Legacy signer/session IDs unavailable; this is not unseen-session or unseen-signer accuracy."
class_weights = compute_class_weight("balanced", classes=np.arange(26), y=y_train)
train_weights = class_weights[y_train].astype(np.float32)
print(f"Validated {len(y)} rows; train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}; {method}")
if limitation:
    print("LIMITATION:", limitation)
print("Training class weights:", dict(zip("ABCDEFGHIJKLMNOPQRSTUVWXYZ", class_weights.round(3))))


Validated 36401 rows; train=25480, val=5460, test=5461; stratified-sample
LIMITATION: Legacy signer/session IDs unavailable; this is not unseen-session or unseen-signer accuracy.
Training class weights: {'A': 0.339, 'B': 1.093, 'C': 2.426, 'D': 2.579, 'E': 0.659, 'F': 1.369, 'G': 0.546, 'H': 1.082, 'I': 1.371, 'J': 0.998, 'K': 0.721, 'L': 0.989, 'M': 1.125, 'N': 0.96, 'O': 2.934, 'P': 2.432, 'Q': 2.016, 'R': 1.088, 'S': 0.854, 'T': 1.12, 'U': 1.184, 'V': 1.141, 'W': 1.075, 'X': 1.279, 'Y': 0.643, 'Z': 0.957}


In [3]:
def batches(features, labels, shuffle=False, weights=None):
    data = tf.data.Dataset.from_tensor_slices((features, labels) if weights is None else (features, labels, weights))
    if shuffle:
        data = data.shuffle(len(features), seed=42)
    options = tf.data.Options()
    options.threading.private_threadpool_size = 1
    return data.batch(128).with_options(options)

train_data = batches(X_train, y_train, shuffle=True, weights=train_weights)
validation_data = batches(X_val, y_val)
model = tf.keras.Sequential([
    tf.keras.layers.Input((42,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(128, activation="mish", kernel_regularizer=tf.keras.regularizers.l2(0.01)),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(64, activation="mish", kernel_regularizer=tf.keras.regularizers.l2(0.01)),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(32, activation="mish", kernel_regularizer=tf.keras.regularizers.l2(0.01)),
    tf.keras.layers.Dense(26, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ batch_normalization             │ (None, 42)             │           168 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         5,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 26)             │           858 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,866 (65.88 KB)

 Trainable params: 16,782 (65.55 KB)

 Non-trainable params: 84 (336.00 B)

In [4]:
history = model.fit(
    train_data, validation_data=validation_data, epochs=1000, verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True)],
)
model.save(model_save_path)
restored = tf.keras.models.load_model(model_save_path)
np.testing.assert_allclose(model(X_val[:128], training=False), restored(X_val[:128], training=False), atol=1e-7)
print(f"Training complete: {len(history.epoch)} epochs; Keras reload verified")


Training complete: 323 epochs; Keras reload verified


In [5]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt

# First use of the untouched test partition: only after training and checkpoint selection.
keras_scores = restored(X_test, training=False).numpy()
keras_predictions = keras_scores.argmax(axis=1)
labels = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
report = classification_report(y_test, keras_predictions, labels=range(26), target_names=labels, zero_division=0, output_dict=True)
print(classification_report(y_test, keras_predictions, labels=range(26), target_names=labels, zero_division=0))
weak_classes = {letter: report[letter] for letter in "UDLT"}
print("WATCH CLASSES (U/D/L/T):", {letter: round(stats["recall"], 3) for letter, stats in weak_classes.items()})
cm = confusion_matrix(y_test, keras_predictions, labels=range(26))
fig, ax = plt.subplots(figsize=(10, 9))
image = ax.imshow(cm, cmap="Blues")
ax.set(xticks=range(26), yticks=range(26), xticklabels=labels, yticklabels=labels, xlabel="Predicted", ylabel="Actual", title="Untouched test confusion matrix")
fig.colorbar(image, ax=ax)
fig.tight_layout()
fig.savefig(MODEL_DIR / "confusion_matrix.png", dpi=140)
plt.close(fig)
pd.DataFrame([{ "letter": letter, **report[letter], "watch_class": letter in "UDLT"} for letter in labels]).to_csv(MODEL_DIR / "test_per_class.csv", index=False)
evaluation = {
    "seed": 42,
    "split_method": method,
    "limitation": limitation,
    "dataset_sha256": hashlib.sha256((MODEL_DIR / "keypoint.csv").read_bytes()).hexdigest(),
    "split_sizes": {"train": len(train_idx), "val": len(val_idx), "test": len(test_idx)},
    "class_counts": dict(zip(labels, np.bincount(y).tolist())),
    "training_class_weights": dict(zip(labels, class_weights.tolist())),
    "epochs": len(history.epoch),
    "keras_test_accuracy": accuracy_score(y_test, keras_predictions),
    "per_class": {letter: report[letter] for letter in labels},
    "watch_classes": list("UDLT"),
}
print(f"Untouched test accuracy: {evaluation['keras_test_accuracy']:.4%}")
np.savez_compressed(MODEL_DIR / "split_manifest.npz", train=train_idx, val=val_idx, test=test_idx, dataset_sha256=evaluation["dataset_sha256"])
evaluation["split_manifest_sha256"] = hashlib.sha256((MODEL_DIR / "split_manifest.npz").read_bytes()).hexdigest()


              precision    recall  f1-score   support

           A       0.81      0.83      0.82       621
           B       0.90      0.92      0.91       192
           C       0.92      0.87      0.89        87
           D       0.89      0.94      0.92        81
           E       0.91      0.82      0.86       318
           F       0.76      0.94      0.84       154
           G       0.87      0.87      0.87       385
           H       0.93      0.94      0.93       194
           I       0.91      0.93      0.92       153
           J       0.85      0.92      0.89       210
           K       0.86      0.68      0.76       292
           L       0.76      0.79      0.77       213
           M       0.68      0.82      0.74       187
           N       0.75      0.89      0.82       219
           O       0.79      0.97      0.87        72
           P       0.94      0.93      0.94        86
           Q       0.76      0.98      0.86       104
           R       0.73    

Untouched test accuracy: 84.1970%


## TFLite export
Freeze inference variables before conversion to avoid the TensorFlow 2.16/Keras 3 variable-conversion failure. Verify all split predictions, not just one example.

In [6]:
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

inference = tf.function(lambda features: restored(features, training=False))
concrete = inference.get_concrete_function(tf.TensorSpec([None, 42], tf.float32))
frozen = convert_variables_to_constants_v2(concrete)
converter = tf.lite.TFLiteConverter.from_concrete_functions([frozen])
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_bytes = converter.convert()
interpreter = tf.lite.Interpreter(model_content=tflite_bytes, num_threads=1)
interpreter.allocate_tensors()
input_info = interpreter.get_input_details()[0]
output_info = interpreter.get_output_details()[0]
assert tuple(input_info["shape"]) == (1, 42)
assert tuple(output_info["shape"]) == (1, 26)
tflite_scores = []
for row in X_test:
    interpreter.set_tensor(input_info["index"], row[None, :].astype(np.float32))
    interpreter.invoke()
    tflite_scores.append(interpreter.get_tensor(output_info["index"])[0].copy())
tflite_scores = np.asarray(tflite_scores)
agreement = float(np.mean(tflite_scores.argmax(axis=1) == keras_predictions))
assert agreement >= 0.99, f"Keras/TFLite agreement {agreement:.4%} below 99%"
assert np.isfinite(tflite_scores).all()
tflite_save_path.write_bytes(tflite_bytes)
print(f"Keras/TFLite agreement: {agreement:.4%}; maximum probability difference: {np.max(np.abs(tflite_scores - keras_scores)):.6f}")
print(f"Exported {len(tflite_bytes)} bytes; Keras and TFLite I/O contracts verified")

evaluation["tflite_test_accuracy"] = accuracy_score(y_test, tflite_scores.argmax(axis=1))
evaluation["keras_tflite_agreement"] = agreement
evaluation["max_probability_difference"] = float(np.max(np.abs(tflite_scores - keras_scores)))
evaluation["model_sha256"] = hashlib.sha256(tflite_bytes).hexdigest()
(MODEL_DIR / "evaluation.json").write_text(json.dumps(evaluation, indent=2) + "\n")
print(f"TFLite untouched test accuracy: {evaluation['tflite_test_accuracy']:.4%}")


Keras/TFLite agreement: 99.3225%; maximum probability difference: 0.079672
Exported 24920 bytes; Keras and TFLite I/O contracts verified
TFLite untouched test accuracy: 84.1238%


W0000 00:00:1791471923.687174  327121 tf_tfl_flatbuffer_helpers.cc:390] Ignored output_format.
W0000 00:00:1791471923.687469  327121 tf_tfl_flatbuffer_helpers.cc:393] Ignored drop_control_dependency.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
